# Output 6 — Probability approach

Excel analogue: **Output 6 - Prob (if applicable)** / **Probability approach**.
Paths are baseline, A1 historical, and the Chart Data most-extreme shock.
Distress probabilities use Excel `NORMDIST` (CPIA, growth, reserves/imports,
remittances, world growth), not `Φ((ratio − T) / T)`.

See `docs/11-scenario.qmd`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from lic_dsf.load import load_core, load_probability, load_rating, load_stress
from lic_dsf.output import (
    external_debt_scenarios_table,
    probabilities_table,
    probability_panel,
)

from lic_dsf.rating import most_extreme_shock_id
from lic_dsf.scenario import (
    ProbabilityAssumptions,
)
from lic_dsf.stress import (
    run_a1_historical_external,
    run_standard_external_stress,
)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent

WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

WORKBOOK


In [ ]:
macro, external, ext_base, _pub_base = load_core(WORKBOOK)
rating = load_rating(WORKBOOK)
stress = load_stress(WORKBOOK)
ci = rating.ci
input6 = stress.input6
residual = stress.residual
external_stress = run_standard_external_stress(macro, external, input6, residual)
historical = run_a1_historical_external(macro, external, residual)

first_proj = int(macro.inputs.first_projection_year)
rating_years = list(range(first_proj, first_proj + 11))
panel_years = [int(y) for y in ext_base.years if int(y) >= first_proj]
mx_sid = most_extreme_shock_id(
    {sid: book.pv_ppg_external_to_gdp() for sid, book in external_stress.items()},
    ci.thresholds.pv_debt_to_gdp,
    rating_years,
)
(
    ci.country,
    ci.thresholds.pv_debt_to_gdp,
    mx_sid,
    panel_years[0],
    panel_years[-1],
    len(panel_years),
)


## Output 6 panels

Excel Output 6 charts plot 11 years; the **Probability approach** tables
run the full projection (`H:AB`, 2024–2044 here). For each indicator we
print two blocks like the workbook sheet:

1. **External debt scenarios** — baseline, A1 historical, and MX shock levels,
   plus the CI threshold and borderline bands (`T × (1 ± bw/2)`).
2. **Probabilities** — Excel `NORMDIST` distress probabilities (percent) and
   the template probability cutoff (`O64:O67`).

The MX shock is Chart Data’s most-extreme selector (PV/GDP, years 2–11),
then the same shock book is used for all four indicators.

In [ ]:
from IPython.display import Markdown, display

INDICATORS = (
    ("PV of debt-to-GDP ratio", "pv_ppg_external_to_gdp", "pv_debt_to_gdp"),
    ("PV of debt-to-exports ratio", "pv_ppg_external_to_exports", "pv_debt_to_exports"),
    (
        "Debt service-to-exports ratio",
        "ppg_debt_service_to_exports",
        "debt_service_to_exports",
    ),
    (
        "Debt service-to-revenue ratio",
        "ppg_debt_service_to_revenue",
        "debt_service_to_revenue",
    ),
)

assumptions = ProbabilityAssumptions(bandwidth=0.1)
covariates = load_probability(WORKBOOK)
thresh = ci.thresholds.as_dict()
mx_book = external_stress[mx_sid]

out_6: dict[str, pd.DataFrame] = {}
for title, method, indicator in INDICATORS:
    panel = probability_panel(
        {
            "baseline": getattr(ext_base, method)().reindex(panel_years),
            "historical": getattr(historical, method)().reindex(panel_years),
            "mx_shock": getattr(mx_book, method)().reindex(panel_years),
        },
        float(thresh[indicator]),
        indicator=indicator,
        assumptions=assumptions,
        covariates=covariates,
    )
    out_6[indicator] = panel
    display(Markdown(f"### {title}"))
    display(Markdown("**External debt scenarios**"))
    display(external_debt_scenarios_table(panel))
    display(Markdown("**Probabilities**"))
    display(probabilities_table(panel))
